In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import numpy as np
import pickle
import os
import matplotlib.pyplot as plt
import wandb
!pip install thop
!pip install tqdm
from thop import profile, clever_format
from torchvision.datasets import CIFAR10
from tqdm import tqdm

class CustomCIFAR10(Dataset):
    def __init__(self, root_dir, train=True, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.data = []
        self.targets = []

        base_folder = os.path.join(root_dir, 'cifar-10-batches-py')

        if train:
            batch_files = [f'data_batch_{i}' for i in range(1, 6)]
        else:
            batch_files = ['test_batch']

        for file_name in batch_files:
            file_path = os.path.join(base_folder, file_name)
            if not os.path.exists(file_path):
                raise RuntimeError(f"Dataset not found at {file_path}")

            with open(file_path, 'rb') as f:
                entry = pickle.load(f, encoding='latin1')
                self.data.append(entry['data'])
                self.targets.extend(entry['labels'])

        self.data = np.vstack(self.data).reshape(-1, 3, 32, 32)
        self.data = self.data.transpose((0, 2, 3, 1))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img, target = self.data[idx], self.targets[idx]
        if self.transform:
            img = self.transform(img)
        return img, target

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class ResNet18_CIFAR(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet18_CIFAR, self).__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(BasicBlock, 64, 2, stride=1)
        self.layer2 = self._make_layer(BasicBlock, 128, 2, stride=2)
        self.layer3 = self._make_layer(BasicBlock, 256, 2, stride=2)
        self.layer4 = self._make_layer(BasicBlock, 512, 2, stride=2)
        self.linear = nn.Linear(512*BasicBlock.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

def plot_grad_flow(named_parameters):
    ave_grads = []
    max_grads = []
    layers = []
    for n, p in named_parameters:
        if(p.requires_grad) and ("bias" not in n) and (p.grad is not None):
            layers.append(n)
            ave_grads.append(p.grad.abs().mean().cpu().item())
            max_grads.append(p.grad.abs().max().cpu().item())

    fig = plt.figure(figsize=(10, 5))
    plt.bar(np.arange(len(max_grads)), max_grads, alpha=0.1, lw=1, color="c", label="Max Gradient")
    plt.bar(np.arange(len(max_grads)), ave_grads, alpha=0.1, lw=1, color="b", label="Average Gradient")
    plt.hlines(0, 0, len(ave_grads)+1, lw=2, color="k" )
    plt.xticks(range(0,len(ave_grads), 1), layers, rotation="vertical")
    plt.xlim(left=0, right=len(ave_grads))
    max_grad_val = max(max_grads) if max_grads else 0.02
    plt.ylim(bottom = -0.001, top=max_grad_val * 1.1 + 0.001)
    plt.xlabel("Layers")
    plt.ylabel("Gradient Magnitude")
    plt.title("Gradient Flow")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    return fig

def plot_weight_update_flow(named_parameters, learning_rate):
    ave_updates = []
    max_updates = []
    layers = []
    for n, p in named_parameters:
        if(p.requires_grad) and ("bias" not in n) and (p.grad is not None):
            layers.append(n)
            update_magnitude = (learning_rate * p.grad).abs()
            ave_updates.append(update_magnitude.mean().cpu().item())
            max_updates.append(update_magnitude.max().cpu().item())

    fig = plt.figure(figsize=(10, 5))
    plt.bar(np.arange(len(max_updates)), max_updates, alpha=0.1, lw=1, color="m", label="Max Update")
    plt.bar(np.arange(len(max_updates)), ave_updates, alpha=0.1, lw=1, color="r", label="Average Update")
    plt.hlines(0, 0, len(ave_updates)+1, lw=2, color="k" )
    plt.xticks(range(0,len(ave_updates), 1), layers, rotation="vertical")
    plt.xlim(left=0, right=len(ave_updates))
    max_update_val = max(max_updates) if max_updates else 0.002
    plt.ylim(bottom = -0.0001, top=max_update_val * 1.1 + 0.0001)
    plt.xlabel("Layers")
    plt.ylabel("Weight Update Magnitude (lr * grad)")
    plt.title("Weight Update Flow")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    return fig

def main():
    config = {
        "epochs": 25,
        "batch_size": 128,
        "lr": 0.1,
        "momentum": 0.9,
        "weight_decay": 5e-4
    }

    wandb.init(project="cifar10-custom-resnet", config=config)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    transform_train = transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    transform_test = transforms.Compose([
        transforms.ToPILImage(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    CIFAR10(root='./data', train=True, download=True)

    train_dataset = CustomCIFAR10(root_dir='./data', train=True, transform=transform_train)
    test_dataset = CustomCIFAR10(root_dir='./data', train=False, transform=transform_test)

    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False, num_workers=2)

    model = ResNet18_CIFAR().to(device)

    input_dummy = torch.randn(1, 3, 32, 32).to(device)
    macs, params = profile(model, inputs=(input_dummy, ), verbose=False)
    flops, params_str = clever_format([macs, params], "%.3f")
    print(f"Model FLOPs: {flops}")
    print(f"Model Params: {params_str}")
    wandb.config.update({"FLOPs": flops, "Params": params_str})

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=config['lr'], momentum=config['momentum'], weight_decay=config['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'])

    wandb.watch(model, log="all", log_freq=100)

    for epoch in range(config['epochs']):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for i, (inputs, targets) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1} Training")):
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()

            if i == 0:
                current_lr = optimizer.param_groups[0]['lr']

                grad_fig = plot_grad_flow(model.named_parameters())
                wandb.log({"Gradient Flow": wandb.Image(grad_fig)}, step=epoch)
                plt.close(grad_fig)

                update_fig = plot_weight_update_flow(model.named_parameters(), current_lr)
                wandb.log({"Weight Update Flow": wandb.Image(update_fig)}, step=epoch)
                plt.close(update_fig)

            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

        train_acc = 100. * correct / total

        model.eval()
        test_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, targets in tqdm(test_loader, desc=f"Epoch {epoch+1} Testing"):
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                test_loss += loss.item()
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()

        acc = 100. * correct / total

        wandb.log({
            "epoch": epoch,
            "train_loss": running_loss / len(train_loader),
            "train_acc": train_acc,
            "test_acc": acc,
            "lr": optimizer.param_groups[0]['lr']
        })

        scheduler.step()
        print(f"Epoch {epoch+1}/{config['epochs']} | Acc: {acc:.2f}% | FLOPs: {flops}")

    wandb.finish()

if __name__ == "__main__":
    main()

Model FLOPs: 557.880M
Model Params: 11.174M


Epoch 1 Testing: 100%|██████████| 100/100 [00:03<00:00, 30.72it/s]


Epoch 1/25 | Acc: 34.85% | FLOPs: 557.880M


Epoch 2 Testing: 100%|██████████| 100/100 [00:04<00:00, 23.88it/s]


Epoch 2/25 | Acc: 45.94% | FLOPs: 557.880M


...

Epoch 24 Testing: 100%|██████████| 100/100 [00:03<00:00, 30.16it/s]


Epoch 24/25 | Acc: 91.85% | FLOPs: 557.880M


Epoch 25 Testing: 100%|██████████| 100/100 [00:03<00:00, 25.17it/s]

Epoch 25/25 | Acc: 91.88% | FLOPs: 557.880M


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
lr,█████▇▇▇▆▆▆▅▅▄▄▃▃▃▂▂▂▁▁▁▁
test_acc,▁▂▃▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇███████
train_acc,▁▃▃▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇███████
train_loss,█▆▅▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
epoch,24
lr,0.00039
test_acc,91.88
train_acc,97.968
train_loss,0.06335
